# M2 parsing benchmark: the whole corpus

Parses the **20,000 assigned papers** (every version) of the five crawl runs into the `elements`, `hierarchy` and `bibtex` Parquet tables on a Colab **CPU** runtime, and measures wall time, RAM and disk. It does not crawl anything.

1. Preflight reads the five runs on Drive and stops unless the crawl is complete (20,000 IDs, no incomplete chunk).
2. Each run is restored from its Drive checkpoint to the local disk **with SHA-256 verification**. This copy is setup and is outside the benchmark.
3. `src.benchmark.m2_benchmark` runs `src.parser.run` over all five runs as one corpus (`--expect-assigned`) with the resource sampler.
4. The state is copied to Drive as the run goes. If the runtime dies, open the notebook on a new runtime, set `RESUME = True` with the **same** `M2_RUN_ID`, and run again: only the work after the last committed shard is repeated. The report keeps the log of every session, as the lecturer asked for multi-session benchmarks.

In [ ]:
# Directory configuration: verify these paths for the mounted Drive.
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"
RUNS = {  # run id -> folder of the run below DRIVE_DIR (each holds data/corpus and work/)
    "only-v1-all": "only-v1/runs/only-v1-all",
    "multi-dang-0000-4999": "multi-version/runs/multi-dang-0000-4999",
    "multi-hieu-0000-4999": "multi-version/runs/multi-hieu-0000-4999",
    "multi-thien-0000-4999": "multi-version/runs/multi-thien-0000-4999",
    "multi-tien-0000-4999": "multi-version/runs/multi-tien-0000-4999",
}
M2_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m2"  # results and resume state go here
LOCAL_ROOT = "/content/m2-local"  # the restored corpus, on the local disk of the runtime

In [ ]:
# Edit this cell. Use a NEW id for a new experiment, the SAME id (and RESUME = True) to resume.
M2_RUN_ID = "m2-full-01"
RESUME = False  # True on a new runtime after the session of this M2_RUN_ID was lost
WORKERS = 2  # the plan: at most 2 processes, under about 4 GB each
SHARD_PAPERS = 100  # papers per Parquet shard (bounds the memory of the parent process)
PAPER_TIMEOUT_S = 600  # a paper that takes longer becomes an 'error' row instead of stalling
SYNC_EVERY_S = 600  # copy the run state to Drive this often
EXPECTED_IDS = 20000
RUN_SCAN = False  # True: gather + references only (a first look at the real corpus)
RUN_M2 = False  # True only after the preflight has been checked
CODE_REF = "main"  # New runs resolve main to an exact SHA; resume uses the saved SHA
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"

In [ ]:
# Prevent a new checkout from being mislabeled while Python keeps old src modules.
import sys

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError(
        "Restart runtime before setup; cached pipeline modules may belong to another revision."
    )

## Authorize Google Drive

Use the Google account that can read the five run folders (add them to My Drive as shortcuts if they are shared) and write `M2_DRIVE_DIR`.

In [ ]:
import json
import os
import shutil
import subprocess
import time

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
if not DRIVE.is_dir():
    raise FileNotFoundError("Check DRIVE_DIR: the team folder is not visible in this Drive")
M2_DRIVE = Path(M2_DRIVE_DIR) / M2_RUN_ID
M2_DRIVE.mkdir(parents=True, exist_ok=True)

In [ ]:
revision = CODE_REF
saved_path = M2_DRIVE / "M2_MANIFEST.json"
if RESUME:
    revision = json.loads(saved_path.read_text())["code_revision"]
elif saved_path.exists():
    raise FileExistsError(f"{saved_path} exists: use a new M2_RUN_ID, or RESUME = True")

In [ ]:
repo_dir = Path("/content/CodeRepo")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", revision], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Implementation revision:", REVISION)

## Install dependencies

Package installation is setup and is outside the benchmark.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "psutil", "pyarrow", "pandas", "tqdm"],
    check=True,
)

## Preflight: is the crawl complete?

Reads `MANIFEST.json` and `work/coverage.json` of each run. The run stops here unless every run is complete, no chunk is incomplete, and the runs together cover exactly the assigned IDs. The runs were crawled with an implementation that dropped every `.bbl`, so about one version in ten has no reference list; the Book states this limitation.

In [ ]:
import pandas as pd

rows = []
for run_id, rel in RUNS.items():
    base = DRIVE / rel
    for name in ("INDEX.json", "MANIFEST.json", "work/coverage.json"):
        if not (base / name).is_file():
            raise FileNotFoundError(f"{base / name} is missing")
    manifest = json.loads((base / "MANIFEST.json").read_text())
    coverage = json.loads((base / "work/coverage.json").read_text())
    rows.append(
        {
            "run": run_id,
            "required_ids": coverage["required_ids"],
            "source_complete": coverage["papers_source_complete"],
            "complete": coverage["complete"],
            "incomplete_chunks": len(coverage["incomplete_chunks"]),
            "source_mode": coverage["source_mode"],
            "revision": manifest["code_revision"][:7],
        }
    )
preflight = pd.DataFrame(rows).set_index("run")
display(preflight)
problems = []
if preflight["required_ids"].sum() != EXPECTED_IDS:
    problems.append(f"{preflight['required_ids'].sum()} IDs in the runs, expected {EXPECTED_IDS}")
if (preflight["source_complete"] != preflight["required_ids"]).any():
    problems.append("a run has papers whose source is not complete")
if (~preflight["complete"]).any() or preflight["incomplete_chunks"].any():
    problems.append("a run is not complete or has incomplete chunks")
if problems:
    raise RuntimeError("; ".join(problems))
print("Crawl complete:", int(preflight["required_ids"].sum()), "papers in", len(RUNS), "runs")

## Record the configuration

A new run writes `M2_MANIFEST.json` to Drive; a resume must use the same revision and settings.

In [ ]:
manifest = {
    "m2_run_id": M2_RUN_ID,
    "code_revision": REVISION,
    "runs": RUNS,
    "workers": WORKERS,
    "shard_papers": SHARD_PAPERS,
    "paper_timeout_s": PAPER_TIMEOUT_S,
    "expected_ids": EXPECTED_IDS,
    "crawl_revisions": sorted(set(preflight["revision"])),
}
if RESUME:
    saved = json.loads(saved_path.read_text())
    for key in ("workers", "shard_papers", "paper_timeout_s", "runs"):
        if saved[key] != manifest[key]:
            raise ValueError(f"RESUME needs the same {key}: saved {saved[key]!r}")
else:
    saved_path.write_text(json.dumps(manifest, indent=2) + "\n")

## Restore the five corpora to the local disk (setup, not benchmarked)

`Checkpoints.restore()` copies each run's `data/` and `work/` to the local disk and checks the SHA-256 of every file against `INDEX.json`, falling back to the preserved object when the visible mirror is ahead of the index. This reads every file once from Drive and can take a while. A run already restored on this runtime is reused.

In [ ]:
from src.benchmark.checkpoints import Checkpoints

started = time.perf_counter()
DATA_ROOTS, CRAWL_WORK_DIRS = [], []
for run_id, rel in RUNS.items():
    local = Path(LOCAL_ROOT) / run_id
    if (local / ".checkpoint-head.json").is_file():
        print(run_id, "already restored on this runtime")
    else:
        shutil.rmtree(local, ignore_errors=True)  # an interrupted restore left a partial copy
        index = Checkpoints(local, DRIVE / rel).restore()
        print(run_id, "restored:", len(index["files"]), "files, generation", index["generation"])
    DATA_ROOTS.append(local / "data" / "corpus")
    CRAWL_WORK_DIRS.append(local / "work")
RESTORE_SECONDS = time.perf_counter() - started
print(f"Restore (setup, outside the benchmark): {RESTORE_SECONDS / 60:.1f} min")
subprocess.run(["df", "-h", "/content"], check=True)

In [ ]:
from src.benchmark.m2_sync import restore_from_drive, sync_to_drive

OUT = Path("/content/m2-out") / M2_RUN_ID
WORK = Path("/content/m2-work") / M2_RUN_ID
REPORT = Path("/content/m2-report") / M2_RUN_ID
if RESUME:
    print("Restored from Drive:", restore_from_drive(M2_DRIVE, OUT, WORK, REPORT))

## Optional first look: gather and references only

Set `RUN_SCAN = True` to see, without writing tables, how many versions are parsed, empty or without a root, and where the parser would have trouble.

In [ ]:
if RUN_SCAN:
    cmd = [
        sys.executable,
        "-m",
        "src.parser.run",
        "--scan",
        "--no-progress",
        "--out-dir",
        "/content/m2-scan-out",
        "--work-dir",
        "/content/m2-scan-work",
    ]
    for root in DATA_ROOTS:
        cmd += ["--data-root", str(root)]
    for work in CRAWL_WORK_DIRS:
        cmd += ["--crawl-work-dir", str(work)]
    subprocess.run(cmd + ["--workers", str(WORKERS), "--expect-assigned"], check=True)
    display(pd.read_csv("/content/m2-scan-work/m2_versions.csv")["status"].value_counts())

## Parse and benchmark

One command for the whole corpus, resumable. The sampler measures the parser and its worker processes; the copy to Drive runs in this notebook every `SYNC_EVERY_S` seconds and is not part of the parser, but it competes for the disk and the network, so the wall time is that of a run that also saves its state. Set `RUN_M2 = True` to start.

In [ ]:
if RUN_M2:
    cmd = [
        sys.executable,
        "-m",
        "src.benchmark.m2_benchmark",
        "--out-dir",
        str(OUT),
        "--work-dir",
        str(WORK),
        "--report",
        str(REPORT),
    ]
    for root in DATA_ROOTS:
        cmd += ["--data-root", str(root)]
    for work in CRAWL_WORK_DIRS:
        cmd += ["--crawl-work-dir", str(work)]
    cmd += [
        "--",
        "--workers",
        str(WORKERS),
        "--shard-papers",
        str(SHARD_PAPERS),
        "--paper-timeout",
        str(PAPER_TIMEOUT_S),
        "--expect-assigned",
    ]
    proc = subprocess.Popen(cmd)
    last = time.monotonic()
    try:
        while proc.poll() is None:
            time.sleep(5)
            if time.monotonic() - last >= SYNC_EVERY_S:
                print("synced to Drive:", sync_to_drive(OUT, WORK, REPORT, M2_DRIVE))
                last = time.monotonic()
    finally:
        proc.wait()
        print(
            "exit code",
            proc.returncode,
            "; final sync:",
            sync_to_drive(OUT, WORK, REPORT, M2_DRIVE),
        )
    if proc.returncode:
        raise RuntimeError(f"the parser exited with {proc.returncode}: see {WORK / 'run.log'}")

## Check the tables and the tree

`check_tables` verifies types, unique IDs, parents and reachability on the whole output; `audit_structure` compares counts in the source with counts in the tree; `audit_sample` draws the 30 random papers (seed 2026) that a team member reads and scores.

In [ ]:
roots = [a for root in DATA_ROOTS for a in ("--data-root", str(root))]
subprocess.run(
    [
        sys.executable,
        "-m",
        "src.qa.check_tables",
        "--out-dir",
        str(OUT),
        "--json",
        str(REPORT / "check_tables.json"),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "src.qa.audit_structure",
        *roots,
        "--out-dir",
        str(OUT),
        "--report",
        str(REPORT),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "src.qa.audit_sample",
        "draw",
        *roots,
        "--out-dir",
        str(OUT),
        "--work-dir",
        str(WORK),
        "--report",
        str(REPORT / "audit30"),
        "-n",
        "30",
        "--seed",
        "2026",
    ],
    check=True,
)

## Export for the Book

`book_csv/` holds the files the Book reads (`data/m2/`). Everything is copied to Drive under `M2_DRIVE_DIR`.

In [ ]:
book_csv = REPORT / "book_csv"
book_csv.mkdir(parents=True, exist_ok=True)
for name in (
    "m2_benchmark_summary.csv",
    "m2_resource_timeseries.csv",
    "m2_papers.csv",
    "m2_versions.csv",
    "structure_audit.csv",
    "m2_sessions.jsonl",
):
    if (REPORT / name).is_file():
        shutil.copy2(REPORT / name, book_csv / name)
for name in ("assigned_check.json", "run_summary.json"):
    if (WORK / name).is_file():
        shutil.copy2(WORK / name, REPORT / name)
print("restore (setup) minutes, if this session did it:", round(RESTORE_SECONDS / 60, 1))
print("final sync:", sync_to_drive(OUT, WORK, REPORT, M2_DRIVE))
shutil.make_archive("/content/m2_evidence", "zip", REPORT)
shutil.copy2("/content/m2_evidence.zip", M2_DRIVE / "m2_evidence.zip")
print("Results on Drive:", M2_DRIVE)